# 📊 Analyse & Comparaison des Modèles — Churn Prediction
Ce notebook entraîne les 4 modèles et génère tous les graphiques nécessaires pour justifier le choix final du modèle.

## 0. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    roc_auc_score, recall_score, f1_score, precision_score,
    classification_report, confusion_matrix, roc_curve
)

# Style global
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('✅ Imports OK')

## 1. Chargement & Préparation des données

In [ ]:
df = pd.read_csv('../SOURCE/dataset.csv')
df['complaint_type'] = df['complaint_type'].fillna('None')

target = 'churn'
X = df.drop(columns=[target, 'customer_id'])
y = df[target]

# Détection automatique des types de colonnes
numeric_features     = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print(f'Shape dataset    : {df.shape}')
print(f'Features num     : {len(numeric_features)}')
print(f'Features cat     : {len(categorical_features)}')
print(f'Taux de churn    : {y.mean():.2%}')

In [ ]:
# Préprocesseur
numeric_transformer     = Pipeline(steps=[('scaler', StandardScaler())])
categorical_transformer = Pipeline(steps=[('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer,     numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Split stratifié
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train : {X_train.shape} | Test : {X_test.shape}')

## 2. Entraînement des 4 modèles

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Random Forest':       RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
    'Deep Learning (MLP)': MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42),
}

results   = {}  # métriques
pipelines = {}  # pipelines entraînés

for name, model in models.items():
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier',   model)
    ])
    pipe.fit(X_train, y_train)

    y_pred  = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    report = classification_report(y_test, y_pred, output_dict=True)

    results[name] = {
        'ROC-AUC':   roc_auc_score(y_test, y_proba),
        'Recall':    recall_score(y_test, y_pred),
        'F1-Score':  f1_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'y_pred':    y_pred,
        'y_proba':   y_proba,
        'report':    report,
    }
    pipelines[name] = pipe
    print(f'{name:30s} | ROC-AUC : {results[name]["ROC-AUC"]:.4f} | Recall : {results[name]["Recall"]:.4f}')

## 3. 📋 Tableau comparatif des métriques

In [ ]:
metrics_df = pd.DataFrame({
    name: {
        'ROC-AUC':   round(v['ROC-AUC'],   4),
        'Recall':    round(v['Recall'],    4),
        'F1-Score':  round(v['F1-Score'],  4),
        'Precision': round(v['Precision'], 4),
    }
    for name, v in results.items()
}).T

# Mise en évidence du meilleur par colonne
display(metrics_df.style
    .highlight_max(axis=0, color='#d4edda')
    .format('{:.4f}')
    .set_caption('Comparaison des métriques — meilleur par colonne surligné en vert')
)

best_model = metrics_df['ROC-AUC'].idxmax()
print(f'\n🏆 Meilleur modèle (ROC-AUC) : {best_model}')

## 4. 📈 Courbes ROC comparatives

In [ ]:
colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd']

plt.figure(figsize=(9, 7))

for (name, v), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, v['y_proba'])
    plt.plot(fpr, tpr, lw=2, color=color,
             label=f"{name} (AUC = {v['ROC-AUC']:.4f})")

plt.plot([0, 1], [0, 1], 'k--', lw=1.2, label='Aléatoire (AUC = 0.50)')
plt.fill_between([0, 1], [0, 1], alpha=0.03, color='grey')

plt.xlabel('Taux de faux positifs (FPR)', fontsize=12)
plt.ylabel('Taux de vrais positifs (TPR)', fontsize=12)
plt.title('Courbes ROC — Comparaison des modèles', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150)
plt.show()
print('✅ Sauvegardé : roc_curves.png')

## 5. 🔲 Matrices de confusion

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, (name, v) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, v['y_pred'])
    cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]  # en %

    # Annotations : chiffres absolus + pourcentages
    labels = np.array([
        [f"{cm[i,j]}\n({cm_pct[i,j]:.1%})" for j in range(2)]
        for i in range(2)
    ])

    sns.heatmap(
        cm, annot=labels, fmt='', cmap='Blues',
        xticklabels=['Prédit 0', 'Prédit 1'],
        yticklabels=['Réel 0',   'Réel 1'],
        ax=ax, linewidths=0.5, cbar=False
    )
    ax.set_title(f"{name}\nRecall={v['Recall']:.3f} | F1={v['F1-Score']:.3f}",
                 fontsize=11, fontweight='bold')
    ax.set_ylabel('Valeur réelle',   fontsize=9)
    ax.set_xlabel('Valeur prédite',  fontsize=9)

plt.suptitle('Matrices de confusion — 4 modèles', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Sauvegardé : confusion_matrices.png')

## 6. 🌲 Feature Importance — Random Forest

In [ ]:
# Récupération du pipeline Random Forest
pipe_rf    = pipelines['Random Forest']
rf_model   = pipe_rf.named_steps['classifier']
prep       = pipe_rf.named_steps['preprocessor']

# Noms des features après OneHot
cat_encoder = prep.named_transformers_['cat'].named_steps['onehot']
cat_cols    = cat_encoder.get_feature_names_out(categorical_features).tolist()
all_features = numeric_features + cat_cols

# Importance
importances = pd.Series(rf_model.feature_importances_, index=all_features)
top15       = importances.sort_values(ascending=True).tail(15)

# Graphique
fig, ax = plt.subplots(figsize=(10, 7))
colors_bar = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(top15)))
top15.plot(kind='barh', ax=ax, color=colors_bar, edgecolor='white')

ax.set_xlabel('Importance relative', fontsize=12)
ax.set_title('Top 15 — Facteurs influençant le Churn (Random Forest)',
             fontsize=13, fontweight='bold')

# Valeurs sur les barres
for bar, val in zip(ax.patches, top15.values):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()
print('✅ Sauvegardé : feature_importance.png')

## 7. 🏆 Justification du choix du modèle

In [ ]:
# Graphique radar / barres comparatives des 4 métriques
metric_keys = ['ROC-AUC', 'Recall', 'F1-Score', 'Precision']
model_names = list(results.keys())

x = np.arange(len(metric_keys))
width = 0.18

fig, ax = plt.subplots(figsize=(13, 6))

for i, (name, color) in enumerate(zip(model_names, colors)):
    vals = [results[name][m] for m in metric_keys]
    bars = ax.bar(x + i * width, vals, width, label=name, color=color, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7.5)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metric_keys, fontsize=12)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Comparaison globale des 4 modèles', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.axhline(y=0.5, color='grey', linestyle='--', lw=0.8, alpha=0.6)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()

# Conclusion
best = metrics_df['ROC-AUC'].idxmax()
print('=' * 60)
print(f'✅ Modèle retenu : {best}')
print(f'   ROC-AUC   : {metrics_df.loc[best, "ROC-AUC"]:.4f}')
print(f'   Recall    : {metrics_df.loc[best, "Recall"]:.4f}')
print(f'   F1-Score  : {metrics_df.loc[best, "F1-Score"]:.4f}')
print(f'   Precision : {metrics_df.loc[best, "Precision"]:.4f}')
print('=' * 60)
print('📁 Graphiques sauvegardés : roc_curves.png, confusion_matrices.png,')
print('                            feature_importance.png, model_comparison.png')